In [ ]:
#Data exploration: look at number of unique entries for some of the columns

categorical_cols = game_df.select_dtypes(include=['object', 'category'])
for col in categorical_cols:
    print(f"--- Unique Values for: {col} ---")
    print(game_df[col].unique())
    print("\n") # Adds a space for readability

In [ ]:
# select relevant variables, excluding Game Title, User Review Text, Release Year
selected_var = ['User Rating', 'Age Group Targeted', 'Price', 'Platform', 'Requires Special Device', 'Developer', 'Publisher', 'Genre','Multiplayer',
            'Game Length (Hours)','Graphics Quality', 'Soundtrack Quality','Story Quality','Game Mode','Min Number of Players']
game_df = game_df[selected_var]

In [ ]:
# List of columns that are categorical with multiple possible categories
multi_cat_cols = ['Age Group Targeted', 'Platform', 'Developer', 'Publisher',
                  'Genre', 'Graphics Quality', 'Soundtrack Quality', 'Story Quality']

# Boolean columns (True/False)
bool_cols = ['Requires Special Device', 'Multiplayer', 'Game Mode']

# ---- STEP 1: Ensure the boolean columns are actually booleans ----
for col in bool_cols:
    game_df[col] = game_df[col].astype(bool)

# ---- STEP 2: Convert boolean columns to integers (0 and 1) ----
# This keeps them usable for machine learning models without expanding into dummy columns.
for col in bool_cols:
    game_df[col] = game_df[col].astype(int)

# ---- STEP 3: Apply one-hot encoding to the multi-category columns ----
game_df = pd.get_dummies(game_df, columns=multi_cat_cols, drop_first=False)

# ---- STEP 4: Inspect the result ----
game_df.head()

In [ ]:
# Separate input and output
y_nonscaled = game_df[['Price']]
X_nonscaled = game_df.drop(columns=['Price'])

# --- Split BEFORE scaling ---
X_train_ns, X_valid_ns, y_train_ns, y_valid_ns = train_test_split(
    X_nonscaled, y_nonscaled, test_size=0.3, random_state=1
)

# --- Fit scalers on training data only ---
scaleInput = MinMaxScaler()
scaleOutput = MinMaxScaler()

X_train = scaleInput.fit_transform(X_train_ns)
X_valid = scaleInput.transform(X_valid_ns)

y_train = scaleOutput.fit_transform(y_train_ns)
y_valid = scaleOutput.transform(y_valid_ns)

# --- Train neural network ---
game_nnet = MLPRegressor(hidden_layer_sizes=(5,), activation='logistic', solver='lbfgs', random_state=1)
game_nnet.fit(X_train, y_train.ravel())

# --- Predict ---
y_pred_scaled = game_nnet.predict(X_valid)

# --- Inverse transform predictions and actual values ---
y_pred = scaleOutput.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_actual = scaleOutput.inverse_transform(y_valid.reshape(-1, 1)).ravel()

# --- Check range ---
print("Min predicted price:", min(y_pred))
print("Max predicted price:", max(y_pred))
print("Min actual price:", min(y_actual))
print("Max actual price:", max(y_actual))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

# --- Regression summary ---
print("Regression Performance Summary")
regressionSummary(y_pred, y_actual)

# --- Compute and print R² score ---
r2 = r2_score(y_actual, y_pred)
print(f"R² Score: {r2:.4f}")

# --- Plot Predicted vs Actual ---
plt.figure(figsize=(7, 6))
plt.scatter(y_actual, y_pred, alpha=0.6, color='dodgerblue', edgecolor='k')
plt.plot([min(y_actual), max(y_actual)], [min(y_actual), max(y_actual)], color='orange', linewidth=2)
plt.title('Predicted vs Actual Game Prices', fontsize=14)
plt.xlabel('Actual Price', fontsize=12)
plt.ylabel('Predicted Price', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
